In [13]:
import sys
sys.path.append("..")
import pandas as pd
from src.data_processing import load_raw_tables

tables = load_raw_tables("../data/raw")

In [14]:
keep = {
 "P_DEMO": ["SEQN", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "RIDEXPRG",
           "INDFMPIR", "WTMECPRP", "SDMVPSU", "SDMVSTRA"],
 "P_DIQ":["SEQN","DIQ010","DIQ050","DIQ070"],
 "P_GHB":["SEQN","LBXGH"],
 "P_BIOPRO":["SEQN","LBXSCR"],
 "P_ALB_CR":["SEQN","URXUMA","URXUCR","URDACT"],
 "P_BMX":["SEQN","BMXBMI","BMXWAIST"],
 "P_BPXO":["SEQN","BPXOSY1","BPXODI1"],
 "P_SMQ":["SEQN","SMQ020","SMQ040"]
}
merged = tables["P_DEMO"][keep["P_DEMO"]].copy()
for name in list(keep)[1:]:
    merged = merged.merge(tables[name][keep[name]], on="SEQN",
                          how="left", validate="one_to_one")
print(merged.shape)
assert len(merged) == 15560

(15560, 24)


In [15]:
adults = merged[merged.RIDAGEYR >= 20].copy()
adults["dm_self"] = adults.DIQ010 == 1
adults["dm_med"] = (adults.DIQ050 == 1) | (adults.DIQ070 == 1)
adults["dm_a1c"] = adults.LBXGH >= 6.5
adults["diabetes"] = adults.dm_self | adults.dm_med | adults.dm_a1c
print("Adults:", len(adults))
print(adults[["dm_self","dm_med","dm_a1c","diabetes"]].sum())
n_only_self = (adults.dm_self & ~adults.dm_med & ~adults.dm_a1c).sum()
n_only_med = (adults.dm_med & ~adults.dm_self & ~adults.dm_a1c).sum()
n_only_a1c = (adults.dm_a1c & ~adults.dm_self & ~adults.dm_med).sum()
print("Only self:", n_only_self, "Only medication:", n_only_med, "Only HbA1c:", n_only_a1c)
pills_no_dx = adults[(adults.DIQ070 == 1) & (adults.DIQ010 != 1)]
print("Pills without diagnosis:", len(pills_no_dx),
      "| HbA1c>=6.5:", pills_no_dx.dm_a1c.sum())

Adults: 9232
dm_self     1420
dm_med      1302
dm_a1c      1171
diabetes    1765
dtype: int64
Only self: 140 Only medication: 61 Only HbA1c: 237
Pills without diagnosis: 108 | HbA1c>=6.5: 47


In [16]:
print(pd.crosstab(adults.DIQ010, adults.DIQ070, dropna=False))

DIQ070   1.0  2.0  9.0   NaN
DIQ010                      
1.0     1016  401    3     0
2.0       52  859    1  6633
3.0       56  205    1     0
9.0        0    2    0     3


In [17]:
dm = adults[adults.diabetes]
print("Cohort so far:", len(dm))

# Pregnancy codes inside the cohort (1 pregnant, 2 not, 3 cannot ascertain, NaN not asked)
print(dm.RIDEXPRG.value_counts(dropna=False))

# Adults with no self-report/medication criterion AND no HbA1c: status unknown
no_self_med = ~(adults.dm_self | adults.dm_med)
print("Unknown status (no self/med, HbA1c missing):", (no_self_med & adults.LBXGH.isna()).sum())

# Cohort members with missing HbA1c, and with zero/missing exam weight
print("Cohort HbA1c missing:", dm.LBXGH.isna().sum())
print("Cohort WTMECPRP missing:", dm.WTMECPRP.isna().sum(), "| equal to 0:", (dm.WTMECPRP == 0).sum())

Cohort so far: 1765
RIDEXPRG
NaN    1661
2.0      89
3.0      10
1.0       5
Name: count, dtype: int64
Unknown status (no self/med, HbA1c missing): 984
Cohort HbA1c missing: 167
Cohort WTMECPRP missing: 0 | equal to 0: 102


In [18]:
# Does RIDSTATR=2 match WTMECPRP>0? (tests my hypothesis)
print(pd.crosstab(adults.RIDSTATR, adults.WTMECPRP > 0, dropna=False))

# Restrict to examined adults and recount
mec = adults[adults.RIDSTATR == 2].copy()
dm_mec = mec[mec.diabetes]
print("Examined adults:", len(mec), "| diabetes cohort:", len(dm_mec))
print("Cohort HbA1c missing:", dm_mec.LBXGH.isna().sum())
print("Unknown status:", ((~(mec.dm_self | mec.dm_med)) & mec.LBXGH.isna()).sum())
print(dm_mec.RIDEXPRG.value_counts(dropna=False))

WTMECPRP  False  True 
RIDSTATR              
1.0         688      0
2.0           0   8544
Examined adults: 8544 | diabetes cohort: 1663
Cohort HbA1c missing: 65
Unknown status: 398
RIDEXPRG
NaN    1565
2.0      89
1.0       5
3.0       4
Name: count, dtype: int64


cohort attrition table and outcome availability

In [19]:
# Final cohort: examined, diabetes, not pregnant and not "cannot ascertain"
cohort = adults[(adults.RIDSTATR == 2) & adults.diabetes
                & ~adults.RIDEXPRG.isin([1, 3])].copy()

# Attrition table
flow = pd.DataFrame({
    "step": ["All participants", "Adults >=20", "Examined (MEC)",
             "Diabetes (any criterion)", "Excluding pregnant/unascertained"],
    "n": [len(merged), len(adults), (adults.RIDSTATR == 2).sum(),
          ((adults.RIDSTATR == 2) & adults.diabetes).sum(), len(cohort)]
})
print(flow)

# Which outcome ingredients are missing in the cohort?
print("Creatinine missing:", cohort.LBXSCR.isna().sum())
print("UACR missing:", cohort.URDACT.isna().sum())
print("Both missing:", (cohort.LBXSCR.isna() & cohort.URDACT.isna()).sum())
print("UACR >= 30:", (cohort.URDACT >= 30).sum())

                               step      n
0                  All participants  15560
1                       Adults >=20   9232
2                    Examined (MEC)   8544
3          Diabetes (any criterion)   1663
4  Excluding pregnant/unascertained   1654
Creatinine missing: 115
UACR missing: 56
Both missing: 15
UACR >= 30: 527


In [21]:
import numpy as np
from src.outcome_definition import egfr_ckdepi_2021, label_outcome

print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([False])))
print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([True])))

labeled = label_outcome(cohort)
print(labeled.kidney_marker.value_counts(dropna=False))
print(labeled.egfr.describe())
print("eGFR < 60:", (labeled.egfr < 60).sum())

[91.69147861]
[68.63349666]
kidney_marker
0.0    896
1.0    662
NaN     96
Name: count, dtype: int64
count    1539.000000
mean       83.766581
std        25.168787
min         2.958234
25%        67.571357
50%        88.156965
75%       102.491755
max       145.145698
Name: egfr, dtype: float64
eGFR < 60: 275


In [22]:
from src.outcome_definition import egfr_ckdepi_2021, label_outcome

print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([False])))
print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([True])))

labeled = label_outcome(cohort)
print(labeled.kidney_marker.value_counts(dropna=False))
print(labeled.egfr.describe())
print("eGFR < 60:", (labeled.egfr < 60).sum())

[91.69147861]
[68.63349666]
kidney_marker
0.0    896
1.0    662
NaN     96
Name: count, dtype: int64
count    1539.000000
mean       83.766581
std        25.168787
min         2.958234
25%        67.571357
50%        88.156965
75%       102.491755
max       145.145698
Name: egfr, dtype: float64
eGFR < 60: 275


In [23]:
# Lowest eGFR rows: are they real?
print(labeled.sort_values("egfr")[["RIDAGEYR","RIAGENDR","LBXSCR","egfr","URDACT"]].head(8))
print("Max creatinine:", labeled.LBXSCR.max())

# Reasons for NaN labels
nan = labeled[labeled.kidney_marker.isna()]
print("Unlabeled:", len(nan))
print("Both missing:", (nan.egfr.isna() & nan.URDACT.isna()).sum())
print("Only creatinine missing (UACR normal):", (nan.egfr.isna() & nan.URDACT.notna()).sum())
print("Only UACR missing (eGFR normal):", (nan.egfr.notna() & nan.URDACT.isna()).sum())

       RIDAGEYR  RIAGENDR  LBXSCR      egfr   URDACT
11393      80.0       1.0   14.97  2.958234  1172.26
5749       72.0       2.0   10.96  3.383306      NaN
742        45.0       2.0   10.60  4.165567  3903.57
9136       66.0       1.0   10.78  4.785974  1555.26
15454      51.0       2.0    7.97  5.650477      NaN
13729      45.0       1.0    8.36  7.399229  5897.81
1359       66.0       1.0    7.43  7.480425      NaN
4568       80.0       1.0    6.75  7.693682  1561.18
Max creatinine: 14.97
Unlabeled: 96
Both missing: 15
Only creatinine missing (UACR normal): 64
Only UACR missing (eGFR normal): 17


In [24]:
# Keep only labelable people for modeling; keep the full labeled table for the report
model_df = labeled[labeled.kidney_marker.notna()].copy()
print(model_df.shape, model_df.kidney_marker.value_counts())

# Flag the sensitivity-analysis group: pills without a diabetes diagnosis
model_df["pills_no_dx"] = (model_df.DIQ070 == 1) & (model_df.DIQ010 != 1)
print("Pills-no-dx in modeling sample:", model_df.pills_no_dx.sum())

model_df.to_csv("../data/interim/cohort_labeled.csv", index=False)

(1558, 30) kidney_marker
0.0    896
1.0    662
Name: count, dtype: int64
Pills-no-dx in modeling sample: 97
